In [ ]:
""" Created on March 28, 2026 // @author: Sarah Shi """

import os
import numpy as np
import pandas as pd

import mineralML as mm

import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'png'

# mineralML Quickstart for Mapped EBSD and/or EDS Data

This notebook shows **how to load and run your quantitative EDS and EBSD data through mineralML** with an example Mount Hood andesite: `MH0811b`. These data were collected for the mineralML manuscript (find the preprint here: https://doi.org/10.31223/X53J2M) and the data are in the GitHub repository (https://github.com/sarahshi/mineralML/tree/main/docs/examples/Maps/MountHood_MH0811b). Please refer to the paper for more information about this sample. The previous example workbook highlights how EDS maps are processed. We will apply the same procedure here, but additionally highlight how the EBSD processing code for CTF files works in mineralML. 

This is a seven step process: 
1. Load and plot a phase map directly from an EBSD CTF file. 
2. Load a directory containing all your CSVs of mapped chemical data with `mm.load_df` (or `pd.read_csv` directly). [Optional] Convert the input from element to oxide wt%.
3. Predict the mineral class with mineralML and automatically plot the mineral phase map, mineral phase counts, and prediction score histograms.
4. Plot EBSD and mineralML-generated EDS maps side-by-side. 
5. Plot prediction score map and individual oxide maps.
6. Plot compositions of mapped minerals in various classification diagrams (ternary, quadrilateral).
7. Plot chemical variation maps.
I have conveniently (I hope!) wrapped all of these bits into one function, called ``mm.run_map`` and ``mm.plot_ctf_phases``

We loaded in the ``mineralML`` Python package as ``mm``. ``mineralML`` has trained machine learning models for classifying minerals. This implementation aims to get your electron microprobe or quantitative EDS compositions classified and processed. We remove some degrees of freedom to simplify the process as much as possible. The minerals considered for this study include: Amphibole, Apatite, Biotite, Calcite, Chlorite, Epidote, Feldspar (Alkali Feldspar and Plagioclase), Garnet, Glass, Kalsilite, Leucite, Melilite, Muscovite, Nepheline, Olivine, Oxide (Rhombohedral_Oxides including Hematite-Ilmenite, Spinel_Group including Magnetite-Spinel), Pyroxene (Clinopyroxene, Orthopyroxene, Na-Pyroxene), Quartz, Rutile, Serpentine, Titanite, Tourmaline, and Zircon. 

CSV files containing your mapped data in oxide weight percentages (or converted to) is necessary. Find an example [here](https://github.com/sarahshi/mineralML/tree/main/docs/examples/Maps/MountHood_MH0811b). The necessary oxides are SiO$_2$, TiO$_2$, Al$_2$O$_3$, FeO$_t$, MnO, MgO, CaO, Na$_2$O, K$_2$O, Cr$_2$O$_3$, P$_2$O$_5$, and ZrO$_2$ (if you are aiming to classify zircon). For the oxides not analyzed for specific minerals, the preprocessing will fill in the nan values as 0. 


## 1. Load and plot a phase map from an EBSD CTF file

In [ ]:
# --- MH0811b Configs ---
mh_file_path = "Maps/MH0811b_EBSD_EDS.ctf" # path to the CTF file exported from AZtec

# Merge more verbose EBSD phase names into broader mineral groups, matching with mineralML naming convention
mh_merge_rules = {
    "Andesine": "Plagioclase", "Orthoclase": "Alkali_Feldspar", # feldspar endmembers to group names
    "Augite": "Clinopyroxene", "Enstatite": "Orthopyroxene", # pyroxene endmembers to group names
    "Magnetite": "Oxide", "Ilmenite": "Oxide", # Fe-Ti oxides to single oxide group
    "Quartz-new": "SiO2_Polymorph", "Cristobalite": "SiO2_Polymorph", # silica phases to single group
}

# Pin each mineral group to a consistent color across figures
mh_base_cols = {
    "Plagioclase": "#66C4C4", "Alkali_Feldspar": "#FEF7C2", "Feldspar_Miscibility_Gap": "#003D36",
    "Clinopyroxene": "#E57A7A", "Orthopyroxene": "#931D1D",
    "Oxide": "#2E2DCE", "Glass": "#F9C300",
    "Apatite": "#5B6768", "SiO2_Polymorph": "#CEC6CD",
    "Unindexed": "#FFFFFF" # white background for unindexed pixels
}

In [ ]:
# Plot the EBSD phase map from the CTF file. This parses the CTF header for
# grid dimensions and phase definitions, maps each pixel's phase ID to its
# name, applies rename_dict for partial case-insensitive matching, and plots
# a 2D categorical phase map with legend ordered by abundance.

mh_ebsd_fig, mh_ebsd_phase_map, _, _, _ = mm.plot_ctf_phases(mh_file_path, # load and plot the CTF phase map
                                                             rename_dict=mh_merge_rules, # apply the merge rules defined above
                                                             phase_colors=mh_base_cols, # apply the color scheme defined above
                                                             title=None, # suppress the auto-generated title
                                                             scalebar_um=100, # 100 µm scale bar, computed from CTF step size
                                                             )

In [ ]:
# Plot a stacked horizontal bar showing area proportions of each phase,
# normalized to classified pixels only. Phases below min_frac are excluded,
# and each segment is annotated with its percentage by default.

fig = mm.plot_phase_proportions(mh_ebsd_phase_map, # input the EBSD phase map to compute proportions
                                title="MH0811b EBSD Phase Proportions", # provide a title
                                min_frac=0.0001, # set a minimum fraction threshold to exclude very rare phases
                                phase_colors=mh_base_cols, # apply the same color scheme as the phase map
                                annotate=True, # annotate each segment with its percentage
                                )

The remaining steps are the same procedure detailed in the first mapping .ipynb on Read The Docs. We will go through it again here for reference, so we can make side by side figures. 

## 2. Load and prepare EDS data for analysis

Here, we will work with EDS data that are in elemental weight percent. This means that we will have to do a conversion to oxide weight perecent.

In [ ]:
# Find your directory of mapped mineral data, stored in Maps/MountHood_MH0811b. 
# This code identifies any file with CSV and appends it to the map. 

base = "Maps"
map_dirs = []
for root, subdirs, files in os.walk(base):
    # Skip any path that includes 'Ignore' in its folder names
    if "Ignore" in root.split(os.sep):
        continue
    
    if any(f.lower().endswith(".csv") for f in files):
        map_dirs.append(root)

print(map_dirs)

## 3. Apply the trained neural network with mm.run_map

We will use ``mm.run_map`` which will return all you need! 

In [ ]:
# Inspect the inputs and outputs of mm.run_map
help(mm.run_map)

In [ ]:
# Here is our all in one function! Read the inputs and outputs provided above. 
 
output = mm.run_map(next((s for s in map_dirs if 'MountHood_MH0811b' in s), None), # provide the directory of interest
                    renormalize=True, # optionally renormalize totals to 100 wt%
                    epoxy_threshold=None, # optionally filter out SiO2 values below a given value, for when EDS picks up epoxy pixels
                    pred_score_threshold=0.6, # provide a prediction score threshold. here, i only want values with >= 0.6 prediction score
                    min_frac=0.001, # provide a minimum pixel fraction for the phase to be displayed
                    units='element_wt%', # provide the unit. can choose 'element_wt%' or 'oxide_wt%'
                    phases=mh_base_cols.keys(), # phases of interest
                    scalebar_um=100, # define size of scalebar desired, in microns
                    pixel_size_um=2.0, # define size of each pixel of scalebar, in microns 
                    scalebar_loc='lower left', # specify location for scalebar
                    scalebar_col='black', # specify color for scalebar
                    phase_colors=mh_base_cols, # provide a color scheme for the phases, as a dictionary mapping phase names to color codes
                    )


In [ ]:
# Plot a stacked horizontal bar showing area proportions of each phase,
# normalized to classified pixels only. Phases below min_frac are excluded,
# and each segment is annotated with its percentage by default.

fig = mm.plot_phase_proportions(output['mineral_map'], # input the EDS phase map to compute proportions
                                title="MH0811b EDS Phase Proportions", # provide a title
                                phases=mh_base_cols.keys(), # specify the phases to include
                                min_frac=0.0001, # set a minimum fraction threshold to exclude very rare phases
                                phase_colors=mh_base_cols, # apply the same color scheme as the phase map
                                annotate=True, # annotate each segment with its percentage
                                )

In [ ]:
# Inspect what is in the outputs
output.keys()

Let's say you now want to work with these data in dataframe form rather than dictionary form. How would you do this? 

In [ ]:
# Pull the dataframe of predictions
df_pred = output['df_pred']
display(df_pred)

## 4. Plot EBSD and mineralML-generated EDS phase maps side-by-side

Let's plot the two phase maps side by side! 

In [ ]:
# EBSD phase map 
mh_ebsd_fig

In [ ]:
# EDS phase map
output['figs'][0]

We can now examine the phase maps produced by the two methods and compare their performance. See the preprint for a more detailed description of these maps. 

## 5. Plot oxide concentration maps and prediction score maps

Let's plot the original oxide maps loaded from the directory. We can examine how well the predicted phase map matches some of the observations made in oxide space. We have a handy function for doing so, with ``mm.plot_oxide_map``.

In [ ]:
fig, ax = mm.plot_oxide_map(
    output, # take the output from run_map
    oxide_name='SiO2', # specify the oxide of interest
    scalebar_um=50, # define size of scalebar desired, in microns
    pixel_size_um=2, # define size of each pixel of scalebar, in microns 
    scalebar_loc='upper right', # specify location for scalebar
    scalebar_col='black', # specify color for scalebar
)

Let's plot the prediction scores from the output, in mapped form. This allows for further investigation to determine where predictions are more and less certain. 

In [ ]:
fig, ax = mm.plot_score_map(
    output, # take the output from run_map
    scalebar_um=50, # define size of scalebar desired, in microns
    pixel_size_um=2, # define size of each pixel of scalebar, in microns 
    scalebar_loc='upper right', # specify location for scalebar
    scalebar_col='black', # specify color for scalebar
)

## 6. Plot compositions of mapped minerals in various classification diagrams (ternary, quadrilateral).

We can do some more with mineralML now. Let's plot all the feldspars, pyroxenes, and spinels in ternary space. 

Identify the phases present.

In [ ]:
# Here are all our feldspars 
fspars = df_pred[df_pred.Predict_Mineral == 'Plagioclase']
display('Feldspars:', fspars)

# Here are all our pyroxenes 
pxs_names = ['Clinopyroxene', 'Orthopyroxene']
pxs = df_pred[df_pred.Predict_Mineral.isin(pxs_names)]
display('Pyroxenes:', pxs)

# Here are all our oxides 
ox_names = ['Oxide']
oxs = df_pred[df_pred.Predict_Mineral.isin(ox_names)]
display('Oxides:', oxs)

Plot these feldspars, pyroxenes, and spinels! 

In [ ]:
# Use FeldsparClassifier to examine at the component space (XAn, XAb, XOr)
fspar_comp = mm.FeldsparClassifier(fspars).calculate_components()
display(fspar_comp)

# Use FeldsparClassifier to plot up these data. 
fig = mm.FeldsparClassifier(fspars).plot()

In [ ]:
# Use PyroxeneClassifier to examine at the component space (En, Wo, Fs). If sodic pyroxenes are also within this input, this will plot them up in the sodic pyroxene ternary
pxs_comp = mm.PyroxeneClassifier(pxs).calculate_components()
display(pxs_comp)

# Use PyroxeneClassifier to plot up these data. 
fig = mm.PyroxeneClassifier(pxs).plot()

In [ ]:
# Use OxideClassifier to examine at the component space.
oxs_comp = mm.OxideClassifier(oxs).calculate_components()
display(oxs_comp)

# Use OxideClassifier to plot up these data. 
fig = mm.OxideClassifier(oxs).plot()

You might note that the structure of these three ``...Classifier`` classes is identical. That is intentional! ``mm.FeldsparClassifier``, ``mm.PyroxeneClassifier``, and ``mm.OxideClassifier`` all have  `calculate_components` and `plot` methods embedded. 

## 7. Plot chemical variation maps

We know the mineralogy now. What if you now want to inspect the chemical variation within the individual crystals? Pull the component maps created for each sample and plot this up with ``mm.plot_component_composite``

This function currently does this calculation for feldspars, pyroxenes, olivines, and amphibole. This can easily be expanded with all the stoichiometric mineral functions. Here, I will just show this for these common igneous phases. 

In [ ]:
# Inspect what’s available:
print(sorted(output["component_maps"].keys()))

#  Plot map highlighting internal compositional variation
fig = mm.plot_component_composite(output, # specify output from above
                                  title="MH0811b", # optionally add a title to this plot
                                  phases=mh_base_cols.keys(), # phases of interest
                                  phase_colors=mh_base_cols, # provide a color scheme for the phases, as a dictionary mapping phase names to color codes
                                  smooth_sigma=0.25, # add a Gaussian blur to smooth compositional data, usually turned off. 
                                  scalebar_um=100, # define size of scalebar desired, in microns
                                  pixel_size_um=2.0, # define size of each pixel of scalebar, in microns 
                                  scalebar_loc='lower left', # specify location for scalebar
                                  scalebar_col='black', # specify color for scalebar
                                  )


One could alternatively use all the functions within mineralML.mapping to do these same things, in a more stepwise manner. Look through the documentation if you would like to use individual bits of this code. 